## Structured LLM output using PydanticAI
- constructing pydanitc model from text input

In [3]:
from pydantic_ai import Agent
from dotenv import load_dotenv
import os

load_dotenv()

#print(f"API: {os.getenv('GOOGLE_GEMINI_API_KEY')}")
agent = Agent(model="google-gla:gemini-2.5-flash")
              #,output_type=EmployeeModel) # 2.5-flash normally

result = await agent.run("Give me an IT employee working in Sweden, keep it short")

result

AgentRunResult(output='**Sofia, a Frontend Developer in Stockholm.** She bikes to her tech job and loves a good fika.')

In [4]:
result.output

'**Sofia, a Frontend Developer in Stockholm.** She bikes to her tech job and loves a good fika.'

In [8]:
from pydantic import BaseModel, Field

class EmployeeModel(BaseModel):
    name: str
    age: int
    salary: int= Field(gt=30_000, lt=50_000)
    position: str

result = await agent.run(
    "Give me an IT employee working in Sweden", output_type=EmployeeModel
)

result

AgentRunResult(output=EmployeeModel(name='Bjorn', age=30, salary=45000, position='Software Engineer'))

In [9]:
employee = result.output
employee

EmployeeModel(name='Bjorn', age=30, salary=45000, position='Software Engineer')

In [11]:
employee.name, employee.age, employee.position

('Bjorn', 30, 'Software Engineer')

In [12]:
employee.model_dump() # for dictionary

{'name': 'Bjorn', 'age': 30, 'salary': 45000, 'position': 'Software Engineer'}

In [16]:
print(employee.model_dump_json(indent=2)) # for json string

{
  "name": "Bjorn",
  "age": 30,
  "salary": 45000,
  "position": "Software Engineer"
}


## List of several employees

In [19]:
result = await agent.run(

    """Give me ten employees in AI and data engineering fields,
    roles can vary, but salary must be between 30000 and 50000
    """,
    output_type=list[EmployeeModel]
)

result.output

[EmployeeModel(name='Alice Smith', age=30, salary=45000, position='AI Engineer'),
 EmployeeModel(name='Bob Johnson', age=32, salary=48000, position='Data Engineer'),
 EmployeeModel(name='Carol Williams', age=28, salary=40000, position='Junior AI Engineer'),
 EmployeeModel(name='David Brown', age=35, salary=49999, position='Senior Data Engineer'),
 EmployeeModel(name='Eve Davis', age=29, salary=38000, position='Machine Learning Engineer'),
 EmployeeModel(name='Frank Miller', age=31, salary=42000, position='Data Scientist (AI Focus)'),
 EmployeeModel(name='Grace Wilson', age=33, salary=47000, position='Big Data Engineer'),
 EmployeeModel(name='Henry Moore', age=27, salary=35000, position='AI Research Assistant'),
 EmployeeModel(name='Ivy Taylor', age=34, salary=49000, position='Data Pipeline Engineer'),
 EmployeeModel(name='Jack Anderson', age=26, salary=39000, position='Junior Machine Learning Engineer')]

In [20]:
employees = result.output
employees

[EmployeeModel(name='Alice Smith', age=30, salary=45000, position='AI Engineer'),
 EmployeeModel(name='Bob Johnson', age=32, salary=48000, position='Data Engineer'),
 EmployeeModel(name='Carol Williams', age=28, salary=40000, position='Junior AI Engineer'),
 EmployeeModel(name='David Brown', age=35, salary=49999, position='Senior Data Engineer'),
 EmployeeModel(name='Eve Davis', age=29, salary=38000, position='Machine Learning Engineer'),
 EmployeeModel(name='Frank Miller', age=31, salary=42000, position='Data Scientist (AI Focus)'),
 EmployeeModel(name='Grace Wilson', age=33, salary=47000, position='Big Data Engineer'),
 EmployeeModel(name='Henry Moore', age=27, salary=35000, position='AI Research Assistant'),
 EmployeeModel(name='Ivy Taylor', age=34, salary=49000, position='Data Pipeline Engineer'),
 EmployeeModel(name='Jack Anderson', age=26, salary=39000, position='Junior Machine Learning Engineer')]

In [22]:
len(employees)

10

In [23]:
for employee in employees:
    print(f"{employee.name = } and {employee.salary}")

employee.name = 'Alice Smith' and 45000
employee.name = 'Bob Johnson' and 48000
employee.name = 'Carol Williams' and 40000
employee.name = 'David Brown' and 49999
employee.name = 'Eve Davis' and 38000
employee.name = 'Frank Miller' and 42000
employee.name = 'Grace Wilson' and 47000
employee.name = 'Henry Moore' and 35000
employee.name = 'Ivy Taylor' and 49000
employee.name = 'Jack Anderson' and 39000


## CV or resume model - a more complex and nested model

In [26]:
class ExperienceModel(BaseModel):
    title: str
    company: str
    description:str
    start_year: int
    end_year: int

class EducationModel(BaseModel):
    title: str
    education_area: str
    school: str
    description:str
    start_year: int
    end_year: int

class CVModel(BaseModel):
    name:str
    age: int
    experience: list[ExperienceModel]
    education: list[EducationModel]

result = await agent.run(
    "Create a Swedish person applying for a data engineering position",
    output_type=CVModel
)

result.output

CVModel(name='Björn Karlsson', age=32, experience=[ExperienceModel(title='Data Engineer', company='Swedbank', description='Developed and maintained data pipelines, built and optimized data warehouses, and implemented ETL processes.', start_year=2018, end_year=2023), ExperienceModel(title='Junior Data Engineer', company='Ericsson', description='Assisted in the development of data infrastructure and supported data-driven projects.', start_year=2016, end_year=2018)], education=[EducationModel(title='M.Sc. Computer Science', education_area='Data Engineering', school='KTH Royal Institute of Technology', description='Specialized in distributed systems and big data technologies.', start_year=2014, end_year=2016), EducationModel(title='B.Sc. Software Engineering', education_area='Computer Science', school='Chalmers University of Technology', description='Focused on software development and algorithms.', start_year=2011, end_year=2014)])

In [27]:
resume = result.output
resume

CVModel(name='Björn Karlsson', age=32, experience=[ExperienceModel(title='Data Engineer', company='Swedbank', description='Developed and maintained data pipelines, built and optimized data warehouses, and implemented ETL processes.', start_year=2018, end_year=2023), ExperienceModel(title='Junior Data Engineer', company='Ericsson', description='Assisted in the development of data infrastructure and supported data-driven projects.', start_year=2016, end_year=2018)], education=[EducationModel(title='M.Sc. Computer Science', education_area='Data Engineering', school='KTH Royal Institute of Technology', description='Specialized in distributed systems and big data technologies.', start_year=2014, end_year=2016), EducationModel(title='B.Sc. Software Engineering', education_area='Computer Science', school='Chalmers University of Technology', description='Focused on software development and algorithms.', start_year=2011, end_year=2014)])

In [28]:
resume.name, resume.age

('Björn Karlsson', 32)

In [30]:
resume.experience[0].title

'Data Engineer'

In [32]:
resume.model_dump()

{'name': 'Björn Karlsson',
 'age': 32,
 'experience': [{'title': 'Data Engineer',
   'company': 'Swedbank',
   'description': 'Developed and maintained data pipelines, built and optimized data warehouses, and implemented ETL processes.',
   'start_year': 2018,
   'end_year': 2023},
  {'title': 'Junior Data Engineer',
   'company': 'Ericsson',
   'description': 'Assisted in the development of data infrastructure and supported data-driven projects.',
   'start_year': 2016,
   'end_year': 2018}],
 'education': [{'title': 'M.Sc. Computer Science',
   'education_area': 'Data Engineering',
   'school': 'KTH Royal Institute of Technology',
   'description': 'Specialized in distributed systems and big data technologies.',
   'start_year': 2014,
   'end_year': 2016},
  {'title': 'B.Sc. Software Engineering',
   'education_area': 'Computer Science',
   'school': 'Chalmers University of Technology',
   'description': 'Focused on software development and algorithms.',
   'start_year': 2011,
   'en

In [34]:
resume.model_dump().keys()

dict_keys(['name', 'age', 'experience', 'education'])

## Optional Post Processing -> load into Duckdb and unnest

In [35]:
import duckdb as db
import dlt

pipeline = dlt.pipeline(
    pipeline_name = "resume_json_duckdb",
    destination=dlt.destinations.duckdb("cv.duckdb"),
    dataset_name="staging"
)

info= pipeline.run(data=[resume.model_dump()], loader_file_format="jsonl", table_name ="cv_entries")
print(info)

Pipeline resume_json_duckdb load step completed in 0.81 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:////Users/susannarokka/Desktop/NBI_year_2/Repo/AI_Engineering_OPA24_Susanna_Rokka/code-alongs/11_2_pydanticai_fundamentals/cv.duckdb location to store data
Load package 1769534106.783529 is LOADED and contains no failed jobs


In [38]:
import duckdb

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__education,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experience,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [39]:

with duckdb.connect("cv.duckdb") as conn:
    desc = conn.sql("desc").df()
    cv_entries = conn.sql("from staging.cv_entries").df() #shift + opt + command -> for writing on several rows
    education= conn.sql("from staging.cv_entries__education").df()
    experience= conn.sql("from staging.cv_entries__experience").df()

desc

,database,schema,name,column_names,column_types,temporary
0,cv,staging,_dlt_loads,"[load_id, schema_name, status, inserted_at, sc...","[VARCHAR, VARCHAR, BIGINT, TIMESTAMP WITH TIME...",False
1,cv,staging,_dlt_pipeline_state,"[version, engine_version, pipeline_name, state...","[BIGINT, BIGINT, VARCHAR, VARCHAR, TIMESTAMP W...",False
2,cv,staging,_dlt_version,"[version, engine_version, inserted_at, schema_...","[BIGINT, BIGINT, TIMESTAMP WITH TIME ZONE, VAR...",False
3,cv,staging,cv_entries,"[name, age, _dlt_load_id, _dlt_id]","[VARCHAR, BIGINT, VARCHAR, VARCHAR]",False
4,cv,staging,cv_entries__education,"[title, education_area, school, description, s...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, BIGINT, B...",False
5,cv,staging,cv_entries__experience,"[title, company, description, start_year, end_...","[VARCHAR, VARCHAR, VARCHAR, BIGINT, BIGINT, VA...",False


In [40]:
cv_entries

,name,age,_dlt_load_id,_dlt_id
0,Björn Karlsson,32,1769534106.783529,jyJ14Xee+hzP5Q


In [41]:
education

,title,education_area,school,description,start_year,end_year,_dlt_parent_id,_dlt_list_idx,_dlt_id
0,M.Sc. Computer Science,Data Engineering,KTH Royal Institute of Technology,Specialized in distributed systems and big dat...,2014,2016,jyJ14Xee+hzP5Q,0,00JwTRjQwaqunw
1,B.Sc. Software Engineering,Computer Science,Chalmers University of Technology,Focused on software development and algorithms.,2011,2014,jyJ14Xee+hzP5Q,1,scxqNnX0UFcDkA


In [44]:
import duckdb

duckdb.sql("""
    SELECT
        name,
        age,
        ex.company,
        ex.description as experience_description,
        ex.start_year as experience_start_year,
        ex.end_year as experience_end_year,
        e.title,
        e.education_area,
        e.school,
        e.start_year as education_start_year,
        e.end_year as education_end_year


    FROM cv_entries cv
    LEFT JOIN education e on cv._dlt_id = e._dlt_parent_id
    LEFT JOIN experience ex on cv._dlt_id = ex._dlt_parent_id
""").df()

,name,age,company,experience_description,experience_start_year,experience_end_year,title,education_area,school,education_start_year,education_end_year
0,Björn Karlsson,32,Ericsson,Assisted in the development of data infrastruc...,2016,2018,M.Sc. Computer Science,Data Engineering,KTH Royal Institute of Technology,2014,2016
1,Björn Karlsson,32,Ericsson,Assisted in the development of data infrastruc...,2016,2018,B.Sc. Software Engineering,Computer Science,Chalmers University of Technology,2011,2014
2,Björn Karlsson,32,Swedbank,"Developed and maintained data pipelines, built...",2018,2023,M.Sc. Computer Science,Data Engineering,KTH Royal Institute of Technology,2014,2016
3,Björn Karlsson,32,Swedbank,"Developed and maintained data pipelines, built...",2018,2023,B.Sc. Software Engineering,Computer Science,Chalmers University of Technology,2011,2014
